# DX 704 Week 12 Project

This week's project will revisit the email spam classifier project from week 9 using large language model embeddings instead of custom features.


The full project description and a template notebook are available on GitHub: [Project 12 Materials](https://github.com/bu-cds-dx704/dx704-project-12).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [1]:
!pip uninstall -y torch torchvision torchaudio
!pip install -q --force-reinstall --no-cache-dir \
    torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 \
    --index-url https://download.pytorch.org/whl/cu130
!pip install -q --upgrade transformers
!pip uninstall -y torch torchvision torchaudio pillow
!pip install -q --no-cache-dir torch==2.10.0 --index-url https://download.pytorch.org/whl/cu130
!pip install -q --upgrade transformers pillow

Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0
ERROR: Could not find a version that satisfies the requirement torch==2.10.0 (from versions: none)
ERROR: No matching distribution found for torch==2.10.0
Found existing installation: pillow 12.1.1
Uninstalling pillow-12.1.1:
  Successfully uninstalled pillow-12.1.1
ERROR: Could not find a version that satisfies the requirement torch==2.10.0 (from versions: none)
ERROR: No matching distribution found for torch==2.10.0


In [2]:
!pip install wget
import wget

In [5]:
import pandas as pd

In [6]:
# pandas can read the zip file directly
enron_spam_data = pd.read_csv("enron_spam_data.zip")
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [7]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

np.float64(0.5092834262664611)

## Part 2: Download BERT Model

We will use a pre-trained BERT model to extract embedding vectors as described in lesson 2.1 this week.
Here is sample code to download the model from [Hugging Face](https://huggingface.co/) and extract one vector.
This model is small enough that you can run it with CPU only, but GPUs will be faster if available.

In [8]:
import torch
from transformers import AutoTokenizer, AutoModel

print("torch:", torch.__version__)
print("cuda version:", torch.version.cuda)
print("gpu available:", torch.cuda.is_available())
print("gpu name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
bert_model = bert_model.to(device)
bert_model.eval()

print("model loaded on", device)

ModuleNotFoundError: No module named 'torch'

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
@torch.no_grad()
def embed_text(text):
    batch = [text]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
    outputs = bert_model(**inputs)
    # CLS token embedding is the first token's hidden state
    cls_emb = outputs.last_hidden_state[:, 0, :]  # shape: [batch_size, 768]
    return cls_emb.cpu()

In [ ]:
v = embed_text("Hi, will you buy my spam?")
v.shape

torch.Size([1, 768])

## Part 3: Create Embedding Vectors

Use BERT to create embeddings for each email in the Enron data set.
You will have to decide how to combine the different columns of the original data set to produce one embedding vector.


Hint: BERT can be run without a GPU, but will be much slower.
Using Google Colab with only a CPU, it runs around 1 embedding per second.
Using Google Colab with the T4 GPU option, it runs around 60 embeddings per second.
Caching is also encouraged to avoid unnecessary reruns.

Save your embeddings in a file "embeddings.tsv.gz" with two columns, Message ID and embedding_vector_json, where embedding_vector_json is a JSON-encoded list.
Make sure that embedding_vector_json is a 1 dimensional list, not 2 dimensional.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [ ]:
# YOUR CHANGES HERE

import json

def combine_email_text(row):
    subject_text = "" if pd.isna(row["Subject"]) else str(row["Subject"])
    message_text = "" if pd.isna(row["Message"]) else str(row["Message"])
    return "Subject: " + subject_text + "\nMessage: " + message_text

enron_spam_data["combined_text"] = enron_spam_data.apply(combine_email_text, axis=1)

@torch.no_grad()
def embed_texts_in_batches(text_list, batch_size=32):
    all_embeddings = []

    for start_index in range(0, len(text_list), batch_size):
        batch_texts = text_list[start_index:start_index + batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        outputs = bert_model(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(cls_embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)

email_text_list = enron_spam_data["combined_text"].tolist()
email_embeddings_tensor = embed_texts_in_batches(email_text_list, batch_size=32)
email_embeddings_array = email_embeddings_tensor.numpy()

embeddings_output = pd.DataFrame({
    "Message ID": enron_spam_data["Message ID"],
    "embedding_vector_json": [json.dumps(vector.tolist()) for vector in email_embeddings_array]
})

embeddings_output.to_csv("embeddings.tsv.gz", sep="\t", index=False)

print(embeddings_output.head())

   Message ID                              embedding_vector_json
0           0  [-0.19050034880638123, -0.31480711698532104, -...
1           1  [-0.5535629391670227, -0.0015751515747979283, ...
2           2  [-0.285826712846756, -0.25143033266067505, -0....
3           3  [-0.7593245506286621, -0.30600371956825256, -0...
4           4  [-0.6594700813293457, -0.20126228034496307, 0....


Submit "embeddings.tsv.gz" in Gradescope.

## Part 4: Train a Linear Regression

Train an ordinary least squares regression for spam/ham status where spam is treated as target value 1 and ham is treated as target value 0 with your embeddings above as the only input variables.


Save the coefficients of your linear model in a file "coefficients.tsv" with columns dim and coefficient where dim specifies the offset in the embedding vector (0-767).
Don't worry about the bias term (but your model should still have it).

In [ ]:
# YOUR CHANGES HERE

from sklearn.linear_model import LinearRegression
import numpy as np

target_values = (enron_spam_data["Spam/Ham"] == "spam").astype(int).to_numpy()

linear_model = LinearRegression()
linear_model.fit(email_embeddings_array, target_values)

coefficients_output = pd.DataFrame({
    "dim": np.arange(email_embeddings_array.shape[1]),
    "coefficient": linear_model.coef_
})

coefficients_output.to_csv("coefficients.tsv", sep="\t", index=False)

print(coefficients_output.head())

   dim  coefficient
0    0     0.056223
1    1    -0.039929
2    2     0.011879
3    3     0.020127
4    4     0.061268


Submit "coefficients.tsv" in Gradescope.

## Part 5: Search for Relevant Documents

The file "queries.tsv" specifies 10 queries.
For each of the queries, encode them as a vector, and find the message that is closest using $L_2$.

Save your results in a file "query-matches.tsv" with columns query_id, query_vector_json, and Message ID.

In [ ]:
# YOUR CHANGES HERE

queries_df = pd.read_csv("queries.tsv", sep="\t")

if "query_id" in queries_df.columns:
    query_id_column = "query_id"
elif "Query ID" in queries_df.columns:
    query_id_column = "Query ID"
else:
    query_id_column = queries_df.columns[0]

possible_query_text_columns = ["query", "Query", "text", "Text"]
query_text_column = None

for column_name in possible_query_text_columns:
    if column_name in queries_df.columns:
        query_text_column = column_name
        break

if query_text_column is None:
    remaining_columns = [col for col in queries_df.columns if col != query_id_column]
    query_text_column = remaining_columns[0]

query_text_list = queries_df[query_text_column].astype(str).tolist()
query_embeddings_tensor = embed_texts_in_batches(query_text_list, batch_size=32)
query_embeddings_array = query_embeddings_tensor.numpy()

matched_message_ids = []

for query_vector in query_embeddings_array:
    distances = np.linalg.norm(email_embeddings_array - query_vector, axis=1)
    closest_index = np.argmin(distances)
    matched_message_ids.append(enron_spam_data.iloc[closest_index]["Message ID"])

query_matches_output = pd.DataFrame({
    "query_id": queries_df[query_id_column],
    "query_vector_json": [json.dumps(vector.tolist()) for vector in query_embeddings_array],
    "Message ID": matched_message_ids
})

query_matches_output.to_csv("query-matches.tsv", sep="\t", index=False)

print(query_matches_output.head())

   query_id                                  query_vector_json  Message ID
0         1  [-0.06660856306552887, 0.4860212206840515, -0....        3018
1         2  [-0.8651905059814453, -0.24639831483364105, 0....        2758
2         3  [-0.2976364195346832, -0.20275232195854187, -0...       18071
3         4  [-0.2151169627904892, 0.19881443679332733, -0....       20009
4         5  [-0.19621729850769043, -0.04857247322797775, -...       15122


Submit "query-matches.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.